# Quick Analysis: Regression Target

Use this notebook to load a dataset, select numeric and categorical features, choose a continuous target, and create descriptive and diagnostic charts.

## How It Works

1. Update the configuration in Section 0.
2. Run the notebook from top to bottom.
3. Review the data quality summary, charts, and feature ranking.

The target must be numeric and contain at least two distinct non-missing values. The charts show associations between each feature and the continuous target.


## 0. Configuration


In [ ]:
# ============================================================
# CONFIGURATION — EDIT THIS SECTION
# ============================================================

USE_DEMO_DATA = True       # Set to False to load your own file
FILE_PATH = ""             # Example: r"C:\data\customers.xlsx" or "/content/data.csv"
SHEET_NAME = 0             # Excel sheet index or name
CSV_SEPARATOR = ","

# Example columns. Replace these with names from your dataset.
NUMERICAL = ["Age", "Income", "Visits"]
CATEGORICAL = ["Region", "Channel", "Membership"]
TARGET = "Spend"           # Continuous numeric outcome

# Limit the number of categories shown in charts
TOP_N_CATEGORIES = 12

# Convert data types when needed
TO_NUMERIC = []
TO_CATEGORICAL = []
TO_DATETIME = []


## 1. Import Libraries


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

## 2. Load Data


In [ ]:
# Create demo data to preview the notebook

if USE_DEMO_DATA:
    rng = np.random.default_rng(42)
    n = 250

    region = rng.choice(
        ["North", "Central", "South"],
        n,
        p=[0.30, 0.25, 0.45]
    )

    channel = rng.choice(
        ["Online", "Store", "Partner"],
        n,
        p=[0.45, 0.40, 0.15]
    )

    membership = rng.choice(
        ["Basic", "Silver", "Gold"],
        n,
        p=[0.50, 0.30, 0.20]
    )

    age = rng.normal(35, 10, n).clip(18, 70)
    income = rng.normal(22_000_000, 7_000_000, n).clip(5_000_000)
    visits = rng.poisson(5, n) + 1

    spend = (
        income * 0.12
        + visits * 220_000
        + np.where(membership == "Gold", 1_500_000, 0)
        + np.where(channel == "Online", 600_000, 0)
        + rng.normal(0, 1_000_000, n)
    ).clip(200_000)

    churn = np.where(
        (visits <= 3) | (membership == "Basic"),
        rng.choice(["Yes", "No"], n, p=[0.55, 0.45]),
        rng.choice(["Yes", "No"], n, p=[0.20, 0.80])
    )

    df = pd.DataFrame({
        "Age": age.round(0),
        "Income": income.round(0),
        "Spend": spend.round(0),
        "Visits": visits,
        "Region": region,
        "Channel": channel,
        "Membership": membership,
        "Churn": churn
    })

    print("Demo data loaded:", df.shape)

In [ ]:
# Load a data file

if not USE_DEMO_DATA:
    file_to_load = FILE_PATH.strip()

    if not file_to_load:
        try:
            from google.colab import files
            uploaded = files.upload()
            file_to_load = next(iter(uploaded.keys()))
        except Exception:
            raise ValueError(
                "Enter FILE_PATH or run this notebook in Google Colab to upload a file."
            )

    suffix = Path(file_to_load).suffix.lower()

    if suffix in [".xlsx", ".xls"]:
        df = pd.read_excel(
            file_to_load,
            sheet_name=SHEET_NAME
        )

    elif suffix == ".csv":
        df = pd.read_csv(
            file_to_load,
            sep=CSV_SEPARATOR
        )

    else:
        raise ValueError(
            "Supported file types are CSV, XLS, and XLSX."
        )

    print("Loaded:", file_to_load)
    print("Shape:", df.shape)

display(df.head())

## 3. Convert Data Types


In [ ]:
# ============================================================
# Convert columns listed in TO_NUMERIC, TO_CATEGORICAL, and TO_DATETIME
# ============================================================

for col in TO_NUMERIC:
    if col in df.columns:
        df[col] = pd.to_numeric(
            df[col],
            errors="coerce"
        )
        print(f"Numeric: {col}")
    else:
        print(f"Column not found: {col}")

for col in TO_CATEGORICAL:
    if col in df.columns:
        df[col] = df[col].astype("category")
        print(f"Categorical: {col}")
    else:
        print(f"Column not found: {col}")

for col in TO_DATETIME:
    if col in df.columns:
        df[col] = pd.to_datetime(
            df[col],
            errors="coerce"
        )
        print(f"Datetime: {col}")
    else:
        print(f"Column not found: {col}")

## 4. Validate Configuration


In [ ]:
# Validate the configuration and continuous target
all_declared = set(NUMERICAL + CATEGORICAL + [TARGET])
missing_columns = sorted(col for col in all_declared if col not in df.columns)
if missing_columns:
    raise ValueError(f"Columns not found: {missing_columns}")
if TARGET in NUMERICAL or TARGET in CATEGORICAL:
    print("The target is also listed as a feature. It will be skipped in target comparison charts.")

df[TARGET] = pd.to_numeric(df[TARGET], errors="coerce")
target_values = df[TARGET].dropna()
if target_values.nunique() < 2:
    raise ValueError("The target must be numeric and contain at least 2 distinct non-missing values.")

print("Configuration is valid")
print("Shape:", df.shape)
print("NUMERICAL:", NUMERICAL)
print("CATEGORICAL:", CATEGORICAL)
print("TARGET:", TARGET)
print("Target summary:")
display(target_values.describe().to_frame(name=TARGET))


# 5. Descriptive Analysis

Run this section to create a distribution chart for each numeric feature and a frequency chart for each categorical feature.


In [ ]:
# ============================================================
# DESCRIPTIVE ANALYSIS — RUN ONCE
# ============================================================

print("=" * 70)
print("DESCRIPTIVE ANALYSIS")
print("=" * 70)

# ------------------------------------------------------------
# 1. NUMERIC FEATURES — DISTRIBUTIONS AND HISTOGRAMS
# ------------------------------------------------------------

for col in NUMERICAL:

    if col not in df.columns:
        continue

    series = pd.to_numeric(
        df[col],
        errors="coerce"
    ).dropna()

    if len(series) == 0:
        print(f"{col}: no valid numeric data.")
        continue

    fig, ax = plt.subplots(figsize=(8, 5))

    ax.hist(
        series,
        bins=20,
        edgecolor="black"
    )

    mean_value = series.mean()
    median_value = series.median()

    ax.axvline(
        mean_value,
        linestyle="--",
        label=f"Mean = {mean_value:,.2f}"
    )

    ax.axvline(
        median_value,
        linestyle=":",
        label=f"Median = {median_value:,.2f}"
    )

    ax.set_title(f"Distribution of {col}")
    ax.set_xlabel(col)
    ax.set_ylabel("Frequency")
    ax.legend()

    plt.tight_layout()
    plt.show()

# ------------------------------------------------------------
# 2. CATEGORICAL FEATURES — BAR CHARTS
# ------------------------------------------------------------

for col in CATEGORICAL:

    if col not in df.columns:
        continue

    counts = (
        df[col]
        .fillna("Missing")
        .astype(str)
        .value_counts()
    )

    if len(counts) > TOP_N_CATEGORIES:
        print(
            f"{col}: {len(counts)} categories. Showing "
            f"the top {TOP_N_CATEGORIES}."
        )
        counts = counts.head(TOP_N_CATEGORIES)

    fig, ax = plt.subplots(figsize=(9, 5))

    counts.sort_values().plot(
        kind="barh",
        ax=ax
    )

    ax.set_title(f"Frequency of {col}")
    ax.set_xlabel("Count")
    ax.set_ylabel(col)

    plt.tight_layout()
    plt.show()

# 6. Diagnostic Analysis

Use these charts to compare each feature with the continuous target:

- Numeric features: inspect scatter plots, trend lines, and Pearson correlation.
- Categorical features: compare target distributions across groups with boxplots.
- Correlation measures linear association. A low value does not rule out a non-linear relationship.


In [ ]:
# Create diagnostic charts for the continuous target
print("=" * 70, "\nDIAGNOSTIC ANALYSIS | target =", TARGET)

# Numeric features: scatter plot, linear trend, and correlation
for col in NUMERICAL:
    if col == TARGET:
        continue
    temp = pd.DataFrame({col: pd.to_numeric(df[col], errors="coerce"), TARGET: df[TARGET]}).dropna()
    if len(temp) < 2 or temp[col].nunique() < 2:
        print(f"Skipping {col}: not enough valid numeric data.")
        continue
    correlation = temp[col].corr(temp[TARGET])
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.scatter(temp[col], temp[TARGET], alpha=0.55)
    slope, intercept = np.polyfit(temp[col], temp[TARGET], 1)
    x_line = np.linspace(temp[col].min(), temp[col].max(), 100)
    ax.plot(x_line, slope * x_line + intercept, color="#D9534F", linestyle="--", label="Linear trend")
    ax.set(title=f"{TARGET} vs {col} | Pearson r = {correlation:.2f}", xlabel=col, ylabel=TARGET)
    ax.legend()
    plt.tight_layout(); plt.show()

# Categorical features: compare target distributions across groups
for col in CATEGORICAL:
    if col == TARGET:
        continue
    temp = pd.DataFrame({col: df[col].astype("object").where(df[col].notna(), "Missing"), TARGET: df[TARGET]}).dropna(subset=[TARGET])
    temp[col] = temp[col].astype(str)
    top = temp[col].value_counts().head(TOP_N_CATEGORIES).index
    temp = temp[temp[col].isin(top)]
    categories = temp[col].value_counts().index.tolist()
    values = [temp.loc[temp[col].eq(category), TARGET].values for category in categories]
    fig, ax = plt.subplots(figsize=(max(9, len(categories) * 0.8), 5))
    ax.boxplot(values, labels=categories, showfliers=False)
    ax.set(title=f"{TARGET} by {col}", xlabel=col, ylabel=TARGET)
    ax.tick_params(axis="x", rotation=45)
    plt.tight_layout(); plt.show()


## 7. Quick Insight Tables

Review missing values, target variation, and simple feature-level associations to decide which variables to investigate further. These summaries are descriptive.


In [ ]:
# QUICK INSIGHTS: missing values and univariate feature ranking
quality = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "missing_n": df.isna().sum(),
    "missing_%": (df.isna().mean() * 100).round(2),
    "unique_n": df.nunique(dropna=True),
}).sort_values("missing_%", ascending=False)
print("Data quality (top missing columns)")
display(quality.head(20))
print("Target summary:")
display(df[TARGET].describe().to_frame(name=TARGET))

rows = []
for col in NUMERICAL:
    if col == TARGET: continue
    temp = pd.DataFrame({"feature": pd.to_numeric(df[col], errors="coerce"), "target": df[TARGET]}).dropna()
    if len(temp) < 2 or temp["feature"].nunique() < 2: continue
    pearson = temp["feature"].corr(temp["target"], method="pearson")
    spearman = temp["feature"].corr(temp["target"], method="spearman")
    rows.append({"feature": col, "type": "numeric", "signal": abs(pearson), "pearson_r": pearson, "spearman_r": spearman, "target_mean_range": np.nan, "valid_n": len(temp), "missing_%": df[col].isna().mean()*100})
for col in CATEGORICAL:
    if col == TARGET: continue
    temp = pd.DataFrame({"group": df[col].astype("object").where(df[col].notna(), "Missing"), "target": df[TARGET]}).dropna(subset=["target"])
    stats = temp.groupby("group", observed=False)["target"].agg(["mean", "size"])
    eligible = stats[stats["size"] >= 5]
    spread = eligible["mean"].max() - eligible["mean"].min() if len(eligible) >= 2 else np.nan
    rows.append({"feature": col, "type": "categorical", "signal": spread, "pearson_r": np.nan, "spearman_r": np.nan, "target_mean_range": spread, "valid_n": int(eligible["size"].sum()) if len(eligible) else 0, "missing_%": df[col].isna().mean()*100})
ranking = pd.DataFrame(rows)
if not ranking.empty:
    ranking = ranking.sort_values("signal", ascending=False, na_position="last").reset_index(drop=True)
    display(ranking)
else:
    print("Not enough data to rank features.")


# 8. How to Read the Charts

- Look for a clear upward or downward pattern in each scatter plot. Curved patterns may indicate a non-linear relationship.
- Pearson correlation summarizes the strength and direction of a linear relationship. Spearman correlation summarizes a monotonic relationship using ranks.
- Compare boxplot medians and spreads across categories. Large differences may indicate an association with the target.
- Check group sizes and missing values before interpreting differences.
- These charts show associations. They do not establish causation or replace model evaluation on validation data.


# 9. How to Use This Notebook

1. Set `USE_DEMO_DATA = False` and enter a file path in `FILE_PATH`. Leave the path empty to select a file in Google Colab.
2. Update `NUMERICAL`, `CATEGORICAL`, and `TARGET` with column names from your dataset.
3. Make sure the target contains numeric values. Add the target column to `TO_NUMERIC` if it is stored as text.
4. Include relevant explanatory features. Exclude identifiers and variables that would reveal the target after the prediction time.
5. Run all cells from top to bottom.


# 10. Notes

- Do not use customer IDs, transaction IDs, or variables created after the prediction time as features.
- Check missing values, outliers, and target distribution before interpreting the charts.
- Charts show up to `TOP_N_CATEGORIES` of the most frequent categories.
- Correlation and group mean differences are exploratory summaries, not causal conclusions or model performance scores.
